# Cyber Log Clustering - Exploration Notebook

This notebook provides an interactive exploration of the UNSW-NB15 dataset and the clustering pipeline results.

## Objectives
1. Understand the dataset structure
2. Explore feature distributions
3. Analyze clustering results
4. Investigate detected anomalies

## Cybersecurity Context
The UNSW-NB15 dataset contains network traffic features from both normal and attack traffic. Our goal is to cluster hosts by their behavioral patterns to identify:
- Compromised machines
- Scanning activity
- Data exfiltration
- Other suspicious behaviors

In [ ]:
# Setup and imports
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from load_data import DataLoader
from feature_engineering import FeatureEngineer
from clustering import ClusteringPipeline
from anomaly_detection import AnomalyDetector
from visualization import Visualizer

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

print("Imports complete!")

## 1. Data Loading and Exploration

In [ ]:
# Load the dataset
DATA_PATH = '/Volumes/Data_IA/UNSW_NB15'

loader = DataLoader(data_path=DATA_PATH)

# Load combined train/test data (faster for exploration)
# Set sample_frac < 1.0 for faster iteration
df = loader.load_combined_dataset(use_raw=False, sample_frac=0.2)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Data summary
summary = loader.get_data_summary()

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Total records: {summary['total_records']:,}")
print(f"Memory usage: {summary['memory_usage_mb']:.2f} MB")
print(f"\nAttack distribution: {summary.get('attack_distribution', 'N/A')}")
print(f"\nTop protocols: {summary.get('protocol_distribution', 'N/A')}")
print(f"\nTop services: {summary.get('service_distribution', 'N/A')}")

In [ ]:
# Preview the data
df.head()

In [ ]:
# Attack category distribution
if 'attack_cat' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    attack_counts = df['attack_cat'].value_counts()
    attack_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title('Distribution of Attack Categories', fontsize=14, fontweight='bold')
    ax.set_xlabel('Attack Category')
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Numeric feature distributions
numeric_cols = ['dur', 'sbytes', 'dbytes', 'spkts', 'dpkts', 'sload', 'dload']
numeric_cols = [c for c in numeric_cols if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols[:8]):
    ax = axes[i]
    # Use log scale for better visualization
    data = df[col].replace(0, np.nan).dropna()
    if len(data) > 0:
        ax.hist(np.log10(data + 1), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_title(f'{col} (log scale)', fontsize=10)
    ax.set_xlabel('log10(value + 1)')

plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
# Create host-level behavioral features
engineer = FeatureEngineer()

# Aggregate by source IP
host_df = engineer.aggregate_by_source_ip(df, min_connections=5)

print(f"Host profiles created: {len(host_df)}")
print(f"Features per host: {len(host_df.columns)}")
host_df.head()

In [ ]:
# Preprocess features
host_df_processed, X_scaled = engineer.preprocess_features(host_df)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Feature names: {engineer.feature_columns[:10]}...")

In [ ]:
# Feature importance by variance
importance_df = engineer.get_feature_importance(X_scaled)

fig, ax = plt.subplots(figsize=(10, 8))
top_n = 15
top_features = importance_df.head(top_n).sort_values('variance')

colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, len(top_features)))
ax.barh(range(len(top_features)), top_features['variance'], color=colors, edgecolor='black')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'])
ax.set_xlabel('Variance')
ax.set_title('Top 15 Features by Variance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of top features
top_feature_names = importance_df.head(12)['feature'].tolist()
top_feature_idx = [engineer.feature_columns.index(f) for f in top_feature_names if f in engineer.feature_columns]

corr_matrix = np.corrcoef(X_scaled[:, top_feature_idx].T)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    xticklabels=top_feature_names,
    yticklabels=top_feature_names,
    cmap='RdBu_r',
    center=0,
    annot=True,
    fmt='.2f',
    ax=ax
)
ax.set_title('Feature Correlation Matrix (Top 12)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Dimensionality Reduction

In [ ]:
# Initialize clustering pipeline
clustering = ClusteringPipeline(random_state=42)

# PCA for initial exploration
X_pca, pca_model = clustering.reduce_dimensions_pca(X_scaled, n_components=2)

print(f"PCA explained variance: {sum(pca_model.explained_variance_ratio_):.2%}")

In [ ]:
# UMAP for better cluster visualization
try:
    X_umap = clustering.reduce_dimensions_umap(X_scaled, n_components=2, n_neighbors=15, min_dist=0.1)
    X_2d = X_umap
    method = 'UMAP'
except Exception as e:
    print(f"UMAP not available: {e}")
    X_2d = X_pca
    method = 'PCA'

print(f"Using {method} for visualization")

In [ ]:
# Visualize 2D projection
fig, ax = plt.subplots(figsize=(12, 8))

# Color by attack ratio if available
if 'attack_ratio' in host_df_processed.columns:
    scatter = ax.scatter(
        X_2d[:, 0], X_2d[:, 1],
        c=host_df_processed['attack_ratio'],
        cmap='RdYlBu_r',
        alpha=0.6,
        s=30
    )
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Attack Traffic Ratio')
else:
    ax.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.6, s=30)

ax.set_xlabel(f'{method} Dimension 1')
ax.set_ylabel(f'{method} Dimension 2')
ax.set_title(f'Host Behavior Space ({method} Projection)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Clustering Analysis

In [ ]:
# Estimate optimal K for KMeans
k_metrics = clustering.estimate_kmeans_k(X_scaled, k_range=(2, 12))

print(f"Optimal K by silhouette: {k_metrics['best_k']}")

In [ ]:
# Elbow plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

k_values = k_metrics['k_values']

# Inertia
axes[0].plot(k_values, k_metrics['inertias'], 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

# Silhouette
axes[1].plot(k_values, k_metrics['silhouettes'], 'go-', linewidth=2, markersize=8)
axes[1].axvline(k_metrics['best_k'], color='red', linestyle='--', label=f"Best K={k_metrics['best_k']}")
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
axes[1].legend()

# Calinski-Harabasz
axes[2].plot(k_values, k_metrics['calinski'], 'ro-', linewidth=2, markersize=8)
axes[2].set_xlabel('Number of Clusters (K)')
axes[2].set_ylabel('Calinski-Harabasz Index')
axes[2].set_title('Calinski-Harabasz Index')

plt.tight_layout()
plt.show()

In [ ]:
# Compare clustering methods
comparison_df = clustering.compare_clustering_methods(
    X_scaled,
    kmeans_k=k_metrics['best_k'],
    hdbscan_min_size=max(10, len(host_df) // 50)
)

comparison_df

In [ ]:
# Use best method (HDBSCAN if available, else KMeans)
if 'hdbscan' in clustering.cluster_labels:
    labels = clustering.cluster_labels['hdbscan']
    method_name = 'HDBSCAN'
else:
    labels = clustering.cluster_labels['kmeans']
    method_name = 'KMeans'

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_outliers = (labels == -1).sum()

print(f"Using {method_name}")
print(f"Clusters: {n_clusters}")
print(f"Outliers: {n_outliers}")

In [ ]:
# Visualize clusters
fig, ax = plt.subplots(figsize=(12, 8))

unique_labels = sorted(set(labels))
colors = sns.color_palette('husl', len(unique_labels))

for i, label in enumerate(unique_labels):
    mask = labels == label
    if label == -1:
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c='red', marker='x', s=50, label=f'Outliers (n={mask.sum()})')
    else:
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=[colors[i]], alpha=0.6, s=30, label=f'Cluster {label} (n={mask.sum()})')

ax.set_xlabel(f'{method} Dimension 1')
ax.set_ylabel(f'{method} Dimension 2')
ax.set_title(f'Host Clusters ({method_name})', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Cluster statistics
cluster_stats = clustering.compute_cluster_statistics(
    X_scaled, labels, engineer.feature_columns, host_df_processed
)

cluster_stats[['cluster', 'size', 'pct_of_total', 'attack_ratio']].round(3)

In [ ]:
# Interpret clusters
interpretations = clustering.interpret_clusters(cluster_stats, engineer.feature_columns)

print("=" * 60)
print("CLUSTER INTERPRETATIONS")
print("=" * 60)
for cluster_id, interpretation in interpretations.items():
    print(f"\nCluster {cluster_id}:")
    print(f"  {interpretation}")

## 5. Anomaly Detection

In [ ]:
# Initialize anomaly detector
detector = AnomalyDetector(contamination=0.05)

# Run multiple detection methods
anomaly_results = detector.detect_all(X_scaled, methods=['isolation_forest', 'lof'])

for method, data in anomaly_results.items():
    pct = data['n_anomalies'] / len(X_scaled) * 100
    print(f"{method}: {data['n_anomalies']} anomalies ({pct:.1f}%)")

In [ ]:
# Ensemble scores
ensemble_scores = anomaly_results['ensemble']['scores']

# Score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(ensemble_scores, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
threshold = np.percentile(ensemble_scores, 95)
axes[0].axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'95th percentile: {threshold:.3f}')
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Number of Hosts')
axes[0].set_title('Anomaly Score Distribution')
axes[0].legend()

# Scatter plot
scatter = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=ensemble_scores, cmap='RdYlBu_r', alpha=0.7, s=30)
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('Anomaly Score')
axes[1].set_xlabel(f'{method} Dimension 1')
axes[1].set_ylabel(f'{method} Dimension 2')
axes[1].set_title('Anomaly Scores in Behavior Space')

plt.tight_layout()
plt.show()

In [ ]:
# Top anomalies
top_anomalies = detector.get_top_anomalies(host_df_processed, ensemble_scores, top_n=20)

# Display with key features
display_cols = ['host_ip', 'anomaly_score', 'anomaly_rank', 'n_connections', 
                'n_unique_dsts', 'bytes_ratio', 'dst_entropy', 'attack_ratio']
display_cols = [c for c in display_cols if c in top_anomalies.columns]

top_anomalies[display_cols].head(10)

In [ ]:
# Compare normal vs anomalous feature distributions
anomaly_stats = detector.compute_anomaly_statistics(
    X_scaled,
    anomaly_results['ensemble']['predictions'],
    engineer.feature_columns
)

print(f"Normal hosts: {anomaly_stats['n_normal']}")
print(f"Anomalous hosts: {anomaly_stats['n_anomalies']}")
print(f"\nTop features differentiating anomalies:")
anomaly_stats['feature_comparison'].head(10)

## 6. Final Analysis

In [ ]:
# Combine all results
results_df = host_df_processed.copy()
results_df['cluster'] = labels
results_df['anomaly_score'] = ensemble_scores
results_df['is_anomaly'] = anomaly_results['ensemble']['predictions'] == -1

print(f"Total hosts: {len(results_df)}")
print(f"Clusters: {n_clusters}")
print(f"Anomalies: {results_df['is_anomaly'].sum()}")

results_df.head()

In [ ]:
# Attack rate by cluster
if 'attack_ratio' in results_df.columns:
    cluster_attack = results_df.groupby('cluster')['attack_ratio'].mean().sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['red' if idx == -1 else 'steelblue' for idx in cluster_attack.index]
    bars = ax.bar(range(len(cluster_attack)), cluster_attack.values * 100, color=colors, edgecolor='black')
    ax.set_xticks(range(len(cluster_attack)))
    ax.set_xticklabels([f'Cluster {c}' if c != -1 else 'Outliers' for c in cluster_attack.index], rotation=45)
    ax.set_ylabel('Attack Traffic %')
    ax.set_title('Attack Rate by Cluster', fontsize=14, fontweight='bold')
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# Save results
results_df.to_csv('../outputs/exploration_results.csv', index=False)
print("Results saved to ../outputs/exploration_results.csv")

## Summary

This notebook explored the UNSW-NB15 dataset and demonstrated:

1. **Data Understanding**: The dataset contains network traffic with various attack types
2. **Feature Engineering**: We created behavioral profiles for each host
3. **Dimensionality Reduction**: UMAP/PCA reveals natural clusters in host behavior
4. **Clustering**: HDBSCAN and KMeans both identify meaningful behavior groups
5. **Anomaly Detection**: Isolation Forest and LOF ensemble identifies suspicious hosts

### Key Findings:
- Clusters correspond to different network behavior types
- Outliers/anomalies often have higher attack traffic ratios
- Features like destination entropy and bytes ratio are key discriminators

### Next Steps:
- Run the full pipeline: `python main.py`
- Explore the Streamlit dashboard: `streamlit run app.py`
- Fine-tune parameters based on SOC requirements